In [1]:
import crystalbuilder as cb

geo = cb.geometry
lattice = cb.lattice
viewer = cb.viewer
bilbao = cb.bilbao

import matplotlib.pyplot as plt
import numpy as np

DEBUG:crystalbuilder.bilbao:Bilbao is logging.
DEBUG:trimesh.util:searching for blender in: C:\Program Files\NVIDIA Corporation\Nsight Compute 2024.1.1;C:\WINDOWS\System32\WindowsPowerShell\v1.0;C:\Program Files\Microsoft MPI\Bin;C:\Python313\Scripts;C:\WINDOWS;C:\Program Files\PowerShell\7;C:\WINDOWS\system32;C:\Program Files\Git\cmd;C:\Program Files\dotnet;C:\Users\Brandon\miniconda3\envs\crystalbuilder-dev;C:\Users\Brandon\miniconda3\condabin;C:\Users\Brandon\miniconda3\envs\crystalbuilder-dev\Library\mingw-w64\bin;C:\Python313;C:\texlive\2024\bin\windows;c:\Users\Brandon\miniconda3\envs\crystalbuilder-dev;C:\Users\Brandon\miniconda3\envs\crystalbuilder-dev\Library\bin;C:\texlive\2024\bin;C:\Program Files\Wolfram Research\WolframScript;.;C:\Program Files\Calibre2;C:\Program Files (x86)\IntelSWToolsMPI\compilers_and_libraries_2020.4.321\windows\mpi\intel64\bin;C:\Program Files\Microsoft VS Code\bin;C:\Users\Brandon\miniconda3\envs\crystalbuilder-dev\Scripts;C:\Users\Brandon\AppData\L

This notebook will show how CrystalBuilder can be used to create arbitrary crystals for visualization or further simulation.

Specifically, let's look at space group #180

This is a hexagonal crystal system with Hermann Mauguin symbol $P6_{2}22$

Interestingly, this is one of the chiral space groups. It is enantiomorphic with space group #181, $P6_{4}22$

In [2]:
a1 = [1/2, -np.sqrt(3)/2, 0] # (1/2)x - (sqrt(3)/2)y
a2 = [1/2, np.sqrt(3)/2, 0] #(1/2)x + (sqrt(3)/2)y
a3 = [0, 0, 1] # cz

a_mag = 1

geo_lattice = lattice.Lattice(a1, a2, a3, magnitude = [a_mag, a_mag, a_mag])

We've defined our lattice, now let's get the positions of the "atoms."

To avoid manually specifying those sites, let's take advantage of the Bilbao Crystallographic Server. Fortunately, CrystalBuilder includes a package that can grab and parse the generators from [the Bilbao site](https://www.cryst.ehu.es). 
<br>
<summary>The Bilbao database is summarized in the following papers, which should be cited if this function is used. </summary>

<details>

>
>M. I. Aroyo, J. M. Perez-Mato, D. Orobengoa, E. Tasci, G. de la Flor, A. Kirov
"Crystallography online: Bilbao Crystallographic Server"
Bulg. Chem. Commun. 43(2) 183-197 (2011).
>
>M. I. Aroyo, J. M. Perez-Mato, C. Capillas, E. Kroumova, S. Ivantchev, G. Madariaga, A. Kirov & H. Wondratschek
"Bilbao Crystallographic Server I: Databases and crystallographic computing programs"
 Z. Krist. 221, 1, 15-27 (2006)
>
>M. I. Aroyo, A. Kirov, C. Capillas, J. M. Perez-Mato & H. Wondratschek
"Bilbao Crystallographic Server II: Representations of crystallographic point groups and space groups"
Acta Cryst. A62, 115-128 (2006)
>
</details>

The bilbao.SpaceGroup() class contains functions for applying symmetry operations to specified points. An instance of the space group must be initialized, with the only argument being the IUCr space group number (1-230).

After this, the calculate_points function will accept any number of input coordinates and perform all of the symmetry operations. The output will be either a numpy array (default) or a list. If using a numpy array, the output will contain only the unique results of the symmetry operations. A list will contain duplicates, as np.unique() requires a conversion to an ndarray. For this reason, it is strongly recommended to use the default numpy output.

<br>

To emphasize the chirality of this space group, let's choose the 3b Wyckoff position (0,0,5/6). The generators 

In [6]:
group = bilbao.SpaceGroup(180)

points = group.calculate_points(point_list=[(0, 0, 5/6)]) #This is the 3b wyckoff position

DEBUG:crystalbuilder.bilbao:Point list is: [(0, 0, 0.8333333333333334)]
DEBUG:crystalbuilder.bilbao:point list type is: <class 'list'>
DEBUG:crystalbuilder.bilbao:individual point in list is type: <class 'tuple'>
DEBUG:crystalbuilder.bilbao:origin: [0.         0.         0.83333333]
Generated 2 points, returned 1. 1 duplicates removed
Returned 1 points


We'll just represent these atoms as spheres. 

In [7]:
radius = .125
atoms = [geo.Sphere(center=point, radius=radius) for point in points]

Now lets use our previously defined lattice to tile these into a crystal.

In [8]:
a1_reps = 2
a2_reps = 2
a3_reps = 2
crystal = geo_lattice.tile_geogeometry(atoms, a1_reps, a2_reps, a3_reps )

DEBUG:crystalbuilder.lattice:Lat: Geometry is a list
DEBUG:crystalbuilder.geometry:Making Structure with Center: [np.float64(-1.0000000000000002), np.float64(0.0), np.float64(-0.16666666666666663)] 
DEBUG:crystalbuilder.geometry:Making Structure with Center: [np.float64(-1.0000000000000002), np.float64(0.0), np.float64(0.8333333333333334)] 
DEBUG:crystalbuilder.geometry:Making Structure with Center: [np.float64(-0.5000000000000001), np.float64(0.8660254037844387), np.float64(-0.16666666666666663)] 
DEBUG:crystalbuilder.geometry:Making Structure with Center: [np.float64(-0.5000000000000001), np.float64(0.8660254037844387), np.float64(0.8333333333333334)] 
DEBUG:crystalbuilder.geometry:Making Structure with Center: [np.float64(-0.5000000000000001), np.float64(-0.8660254037844387), np.float64(-0.16666666666666663)] 
DEBUG:crystalbuilder.geometry:Making Structure with Center: [np.float64(-0.5000000000000001), np.float64(-0.8660254037844387), np.float64(0.8333333333333334)] 
DEBUG:crystalbu

We can visualize this using CrystalBuilder's viewer package, which builds the structure as a Vedo scene. 

In [9]:
scene = viewer.visualize(crystal)
scene.show().close()

mode:vedo


This looks pretty boring. We can choose a different wyckoff site, but let's go ahead and compare it to #181.

In [10]:
other_group = bilbao.SpaceGroup(181)
other_points = other_group.calculate_points(point_list=[(0, 0, 5/6)]) #This is the 3b wyckoff position
other_radius = .125
other_atoms = [geo.Sphere(center=point, radius=other_radius) for point in other_points]

a1_reps = 2
a2_reps = 2
a3_reps = 2
other_crystal = geo_lattice.tile_geogeometry(other_atoms, a1_reps, a2_reps, a3_reps )

DEBUG:crystalbuilder.bilbao:Point list is: [(0, 0, 0.8333333333333334)]
DEBUG:crystalbuilder.bilbao:point list type is: <class 'list'>
DEBUG:crystalbuilder.bilbao:individual point in list is type: <class 'tuple'>
DEBUG:crystalbuilder.bilbao:origin: [0.         0.         0.83333333]
Generated 2 points, returned 1. 1 duplicates removed
Returned 1 points
DEBUG:crystalbuilder.lattice:Lat: Geometry is a list
DEBUG:crystalbuilder.geometry:Making Structure with Center: [np.float64(-1.0000000000000002), np.float64(0.0), np.float64(-0.16666666666666663)] 
DEBUG:crystalbuilder.geometry:Making Structure with Center: [np.float64(-1.0000000000000002), np.float64(0.0), np.float64(0.8333333333333334)] 
DEBUG:crystalbuilder.geometry:Making Structure with Center: [np.float64(-0.5000000000000001), np.float64(0.8660254037844387), np.float64(-0.16666666666666663)] 
DEBUG:crystalbuilder.geometry:Making Structure with Center: [np.float64(-0.5000000000000001), np.float64(0.8660254037844387), np.float64(0.83

And let's add it to our visualizer and display everything again.

In [11]:
scene = viewer.visualize(crystal, c='blue', alpha=.5)
viewer.add_to_visualizer(other_crystal, scene, c='red', alpha=.5)
scene.show().close()

mode:vedo
